# Fieldtrip: Audio vs Motor Spatio-temporal clustering

Lead authors: Hadi Zaatiti <hz3752@nyu.edu>

- Description of experiment
The `Audio vs Visual vs Motor` code experiment in `Psychtoolbox` can be found here:

[Auditory vs Visual vs Motor](https://github.com/Hzaatiti/meg-pipeline/blob/main/experiments/psychtoolbox/auditory-vs-visual/auditory_vs_visual.m)

## Importing data 


The data used in this notebook is hosted on `NYU BOX`. Permissions are given upon request.

- Install the BOX app from [here](https://www.box.com/resources/downloads)
- Set an environment variable with name `MEG_DATA` to the path of the Data folder e.g.,
    - `C:\Users\user_name\Box\MEG\Data`
    - or `C:\Users\user_name\Box\Data

## MATLAB setup


Make sure that:
- Fieldtrip is installed in MATLAB
- Add to MATLAB path the custom-made functions for NYUAD MEG lab found [here](https://github.com/BioMedicalImaging-Core-NYUAD/neurowaves-lab-documentation/tree/main/pipeline/field_trip_pipelines/matlab_functions)  


## Spatio-temporal clustering using Fieldtrip


Reference tutorial applied in this notebook: https://www.fieldtriptoolbox.org/tutorial/stats/cluster_permutation_timelock/

In this notebook, we will apply spatio-temporal clustering on a within-subject data. The reader is supposed to have understood spatio-temporal clustering technique, the notebook is meant for applying such algorithm on an NYUAD acquired dataset and not detailing the method itself.

### Importing visual and audio trials

Let's start off by defining the path to our MEG pre-processed trials

In [14]:
% Read the environment variable to NYU BOX
MEG_DATA_FOLDER = getenv('MEG_DATA');


% Define paths
TASK_NAME = 'audio-visual-motor';
SYSTEM = 'meg';
SUB_ID = 'sub-001';
SESSION_ID = 'ses-01';
DERIVATIVES = 'derivatives';
PIPELINE = 'fieldtrip_trials_preprocessed';

% Construct the directory path
TRIALS_FOLDER_PATH = fullfile(MEG_DATA_FOLDER, TASK_NAME, DERIVATIVES, PIPELINE, SUB_ID, SESSION_ID);


Load the visual and motor trials into the memory then display their shape.
The visual trials correspond to a strong visual flash while the motor trials correspond to a button press.
The trials are output of `ft_preprocessing` or similar structured output

In [15]:
load(fullfile(TRIALS_FOLDER_PATH, 'trials_visual.mat'));
load(fullfile(TRIALS_FOLDER_PATH, 'trials_motor.mat'));

trials_visual
trials_motor


trials_visual = 

  struct with fields:

         label: {207×1 cell}
     trialinfo: [141×1 double]
    sampleinfo: [141×2 double]
          grad: [1×1 struct]
         trial: {1×141 cell}
          time: {1×141 cell}
       fsample: 1000
           cfg: [1×1 struct]


trials_motor = 

  struct with fields:

         label: {207×1 cell}
     trialinfo: [146×1 double]
    sampleinfo: [146×2 double]
          grad: [1×1 struct]
         trial: {1×146 cell}
          time: {1×146 cell}
       fsample: 1000
           cfg: [1×1 struct]




Spatio-temporal clustering within subject, will compare the MEG measurements for each sample pair (channel, time point) across conditions.
In this notebook we are considering two conditions, the visual one and motor one.
The comparison is based on attempting to find "significant" clusters selected from samples selected, themselves, according to a statistical t-test.

The statistical t-test is performed for each sample pair (channel, time point) across conditions and requires knowing the mean of the measurement across trials from condition 1 and the mean across condition 2, aswell as the spread over each group of trials.

Let us compute the averaged trials, while keeping the measurement of each trial.

In [16]:
cfg = [];
cfg.keeptrials = 'yes';

timelock_visual = ft_timelockanalysis(cfg, trials_visual)
timelock_motor = ft_timelockanalysis(cfg, trials_motor)

the input is raw data with 207 channels and 141 trials
the call to "ft_selectdata" took 0 seconds
the call to "ft_timelockanalysis" took 2 seconds

timelock_visual = 

  struct with fields:

          time: [-0.5000 -0.4990 -0.4980 -0.4970 -0.4960 -0.4950 -0.4940 -0.4930 -0.4920 -0.4910 -0.4900 -0.4890 -0.4880 -0.4870 -0.4860 -0.4850 -0.4840 -0.4830 -0.4820 -0.4810 -0.4800 -0.4790 … ] (1×1700 double)
         label: {207×1 cell}
          grad: [1×1 struct]
    sampleinfo: [141×2 double]
         trial: [141×207×1700 double]
     trialinfo: [141×1 double]
        dimord: 'rpt_chan_time'
           cfg: [1×1 struct]

the input is raw data with 207 channels and 146 trials
the call to "ft_selectdata" took 0 seconds
the call to "ft_timelockanalysis" took 2 seconds

timelock_motor = 

  struct with fields:

          time: [-0.5000 -0.4990 -0.4980 -0.4970 -0.4960 -0.4950 -0.4940 -0.4930 -0.4920 -0.4910 -0.4900 -0.4890 -0.4880 -0.4870 -0.4860 -0.4850 -0.4840 -0.4830 -0.4820 -0.4810 -0.4800 -

We will not define the spatio-temporal clustering parameters, the full details of how these parameters affect the analysis should be understood from the referenced fieldtrip tutorial and the original referenced papers.

In [17]:
cfg = [];
cfg.method='montecarlo'; % we will define a certain number of permutation and perform purely randomly number of permutations
cfg.statistic = 'indepsamplesT'; % t-value is attributed per sample

cfg.correctm = 'cluster';
cfg.clusteralpha = 0.05;   % threshold level for identifying "good" samples with best t-values
cfg.clusterstatistic= 'maxsum';
cfg.minnbchan = 2;  % Minimum number of channels that are in the neighborhood of a sample, to be included in the clustering algorithm
% (It will still have to pass the alpha threshold constraint)

cfg.tail = 0;  % one-sided or two sided test
cfg.clusterail=0;

% Neighbours prepare
ncfg = [];
ncfg.method = 'distance';
ncfg.grad = timelock_visual.grad;
neighbours = ft_prepare_neighbours(ncfg);

cfg.neighbours = neighbours;
cfg.alpha = 0.025;  % threshold of the permutation test (not exactly sure what that is)
cfg.numrandomization = 10000;

n_visual = size(timelock_visual.trial, 1);

n_motor = size(timelock_motor.trial, 1);

cfg.design           = [ones(1,n_visual), ones(1,n_motor)*2]; % design matrix
cfg.ivar             = 1; % number or list with indices indicating the independent variable(s)

cfg.channel       = {'AG*'};     % cell-array with selected channel labels
cfg.latency       = [0 1];       % time interval over which the experimental
                                 % conditions must be compared (in seconds)

using gradiometers specified in the configuration
using a distance threshold of 4
there are on average 6.2 neighbours per channel
the call to "ft_prepare_neighbours" took 1 seconds



Execute the statistics computation. We picked a high number for the permutation (cfg.numrandomization) to get a better approximation of the p-value.
This will take a while, for testing purposes go with a small number of permutation (e.g., 100) then attempt a higher one.

In [ ]:
[stat] = ft_timelockstatistics(cfg, timelock_visual, timelock_motor);


save stat_visual_motor stat;